# 결정 구조 실습

**Crystal Structure**

원자들의 주기적인 공간 배열. 조성이 같아도 배열에 따라 물성이 달라질 수 있다.

소재 분야에서 이해하기: 같은 원소 조성의 서로 다른 구조를 비교한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 격자를 직접 만들어 봅니다

같은 조성이라도 배열이 다르면 밀도와 이웃 수가 달라집니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def lattice(kind, repeat=3, a=1.0):
    """단순입방(sc), 체심입방(bcc), 면심입방(fcc) 격자의 원자 좌표."""
    basis = {'sc': [(0, 0, 0)],
             'bcc': [(0, 0, 0), (0.5, 0.5, 0.5)],
             'fcc': [(0, 0, 0), (0.5, 0.5, 0), (0.5, 0, 0.5), (0, 0.5, 0.5)]}[kind]
    points = []
    for i in range(repeat):
        for j in range(repeat):
            for k in range(repeat):
                for bx, by, bz in basis:
                    points.append(((i + bx) * a, (j + by) * a, (k + bz) * a))
    return np.array(points)

for kind in ('sc', 'bcc', 'fcc'):
    print('%s: 단위포당 원자 %d개' % (kind, {'sc': 1, 'bcc': 2, 'fcc': 4}[kind]))

In [ ]:
from scipy.spatial import cKDTree

print('%-5s %-14s %-12s %s' % ('type', 'nearest dist', 'coordination', 'packing fraction'))
for kind, atoms_per_cell in [('sc', 1), ('bcc', 2), ('fcc', 4)]:
    points = lattice(kind, repeat=5)
    tree = cKDTree(points)
    centre = points[np.argmin(np.linalg.norm(points - points.mean(0), axis=1))]
    distances, _ = tree.query(centre, k=15)
    nearest = distances[1]
    coordination = int(np.sum(np.abs(distances - nearest) < 1e-6))
    radius = nearest / 2
    packing = atoms_per_cell * (4 / 3) * np.pi * radius ** 3 / 1.0
    print('%-5s %-14.4f %-12d %.4f' % (kind, nearest, coordination, packing))
print('\n이론값: sc 0.524 / bcc 0.680 / fcc 0.740')

In [ ]:
figure = plt.figure(figsize=(11, 3.6))
for index, kind in enumerate(('sc', 'bcc', 'fcc'), 1):
    axis = figure.add_subplot(1, 3, index, projection='3d')
    points = lattice(kind, repeat=2)
    axis.scatter(points[:, 0], points[:, 1], points[:, 2], s=40)
    axis.set_title(kind); axis.set_xticks([]); axis.set_yticks([]); axis.set_zticks([])
plt.tight_layout(); plt.show()

## 2. 해석

원자 배열이 달라지면 최근접 거리·배위수·충전율이 모두 달라지고, 이것이 밀도·확산·소성 거동으로
이어집니다. 조성만으로 물성을 예측하려는 모델이 놓치는 정보가 바로 이 구조입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#crystal)을 여세요.